# Test Silver — WHO
Verifica las 2 tablas Silver de WHO sin escanear datos completos.
Usa `limit(10)` — no ejecuta `count()` ni agregaciones costosas.

In [ ]:
from pyspark.sql import functions as F

SILVER_SCHEMA = "stage_silver_ss2"
_TABLES = ["who_guatemala", "who_costa_rica"]

# Columnas Bronze que NO deben aparecer en Silver
_BRONZE_LEAKED = {
    "indicator_code", "indicator_name", "number", "year", "sex",
    "age_group_code", "age_group", "country",
    "percentage_of_cause_specific_deaths_out_of_total_deaths",
    "age_standardized_death_rate_per_100_000_standard_population",
    "death_rate_per_100_000_population",
    "volume_path", "gdrive_file_id",
    "cf", "rate_per_100k", "death_rate",  # nombres intermedios descartados
}

---
## who_guatemala

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.who_guatemala")
df.printSchema()

In [ ]:
display(df.limit(10))

---
## who_costa_rica

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.who_costa_rica")
df.printSchema()

In [ ]:
display(df.limit(10))

---
## Chequeos de calidad (sin full scan)

In [ ]:
# 1. Columnas Bronze no deben filtrarse a Silver
print("── Columnas Bronze no deben aparecer en Silver ──")
for t in _TABLES:
    leaked = set(spark.table(f"{SILVER_SCHEMA}.{t}").columns) & _BRONZE_LEAKED
    print(f"  {t}: {'OK' if not leaked else f'WARN — columnas presentes: {leaked}'}")

In [ ]:
# 2. Tipos de columnas clave
print("── Tipos de columnas clave ──")
_EXPECTED_TYPES = {"anio": "int", "defunciones": "bigint"}
for t in _TABLES:
    schema_map = {f.name: f.dataType.simpleString() for f in spark.table(f"{SILVER_SCHEMA}.{t}").schema}
    for col, expected in _EXPECTED_TYPES.items():
        actual = schema_map.get(col, "NOT FOUND")
        status = "OK" if actual == expected else f"WARN ({actual})"
        print(f"  {t}.{col}: {actual} [{status}]")

In [ ]:
# 3. sexo solo debe tener F, M, Ambos
print("── Valores de sexo (debe ser solo F, M, Ambos) ──")
_VALID_SEXO = {"F", "M", "Ambos"}
for t in _TABLES:
    vals = [r.sexo for r in spark.table(f"{SILVER_SCHEMA}.{t}").select("sexo").distinct().collect()]
    unexpected = [v for v in vals if v not in _VALID_SEXO]
    print(f"  {t}: {sorted(vals)}  {'OK' if not unexpected else f'WARN inesperados: {unexpected}'}")

In [ ]:
# 4. grupo_etario no debe tener prefijo 'Age' ni guión bajo residual
print("── grupo_etario normalizado ──")
for t in _TABLES:
    vals = [r.grupo_etario for r in spark.table(f"{SILVER_SCHEMA}.{t}").select("grupo_etario").distinct().limit(40).collect()]
    malformed = [v for v in vals if v and v.startswith("Age")]
    print(f"  {t}: {'OK' if not malformed else f'WARN sin normalizar: {malformed}'}")

In [ ]:
# 5. NULLs en tasas deben existir (no se imputan con 0)
print("── NULLs en tasas (deben existir) ──")
for t in _TABLES:
    s = spark.table(f"{SILVER_SCHEMA}.{t}").limit(1000)
    print(f"  {t} (muestra 1k):")
    print(f"    pct_causa_total_muertes nulls : {s.filter(F.col('pct_causa_total_muertes').isNull()).count()}")
    print(f"    tasa_estandarizada_x100k nulls: {s.filter(F.col('tasa_estandarizada_x100k').isNull()).count()}")
    print(f"    tasa_mortalidad_x100k nulls   : {s.filter(F.col('tasa_mortalidad_x100k').isNull()).count()}")

In [ ]:
# 6. nombre_causa debe estar en español — spot-check de 3 códigos conocidos
print("── Traducción nombre_causa → español (spot-check) ──")
_SPOT = {
    "CG0000": "Todas las causas",
    "CG0395": "COVID-19",
    "CG1040": "Enfermedades cardiovasculares",
}
for t in _TABLES:
    sample = (
        spark.table(f"{SILVER_SCHEMA}.{t}")
        .select("codigo_causa", "nombre_causa")
        .distinct()
        .limit(500)
    )
    print("")
    print(f"  {t}:")
    for code, expected in _SPOT.items():
        row = sample.filter(F.col("codigo_causa") == code).first()
        if row is None:
            print(f"    {code}: NO ENCONTRADO en muestra")
        else:
            status = "OK" if row.nombre_causa == expected else f"WARN — got '{row.nombre_causa}', esperado '{expected}'"
            print(f"    {code}: {row.nombre_causa}  [{status}]")
    # Verificar que no quedan nombres en inglés (buscar 'Disease' o 'Causes' como señal)
    english_leak = sample.filter(
        F.col("nombre_causa").rlike(r'\b(Disease|Causes|Infections|Cancer|Disorders)\b')
    ).count()
    print(f"    Nombres en inglés detectados: {english_leak}  (debe ser 0)")